# Proyecto OilyGiant: Seleccion de regiones para nuevos pozos

**Objetivo ejecutivo**
Definir la region con mejor retorno esperado y riesgo controlado, usando un modelo lineal simple y un analisis de incertidumbre con bootstrapping.

**Que se entrega**
- Comparativo de desempeno del modelo por region
- Estimacion de beneficio potencial con los 200 mejores pozos
- Riesgo de perdida y recomendacion final

## Reglas del negocio y criterios de decision
- 200 pozos a desarrollar; presupuesto total: 100,000,000 USD
- Ingreso por unidad de producto: 4,500 USD (product en miles de barriles)
- Metodo de entrenamiento: regresion lineal
- Evaluacion operativa: muestreo de 500 pozos y seleccion de los mejores 200
- Criterio final: riesgo de perdida < 2.5% y mayor beneficio promedio

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
BUDGET = 100_000_000
N_WELLS = 200
REVENUE_PER_UNIT = 4_500
SAMPLE_SIZE = 500
RANDOM_STATE = 12345
BOOTSTRAP_SAMPLES = 1000

MIN_PRODUCT_PER_WELL = BUDGET / N_WELLS / REVENUE_PER_UNIT
MIN_PRODUCT_PER_WELL

111.11111111111111

**Interpretacion ejecutiva**
El umbral minimo por pozo es 111.11 (miles de barriles). Esta cifra es la referencia operativa para decidir si una region puede sostener el plan sin depender exclusivamente de los pozos top.

In [3]:
root = Path.cwd().resolve()
data_dir = root / "data" if (root / "data").exists() else root.parent / "data"

files = {
    "region_0": data_dir / "geo_data_0.csv",
    "region_1": data_dir / "geo_data_1.csv",
    "region_2": data_dir / "geo_data_2.csv",
}

data = {name: pd.read_csv(path) for name, path in files.items()}
{key: df.shape for key, df in data.items()}

{'region_0': (100000, 5), 'region_1': (100000, 5), 'region_2': (100000, 5)}

## Revision inicial de datos
Validamos estructura, faltantes y duplicados para asegurar que el modelo trabaje con datos consistentes y sin sesgos operativos evidentes.

In [4]:
def profile_df(name, df):
    print(f"\n{name}")
    print("-" * len(name))
    print("shape:", df.shape)
    print("missing:", df.isna().sum().sum())
    print("duplicates:", df.duplicated().sum())
    print("columns:", list(df.columns))
    print(df.head(3))

for name, df in data.items():
    profile_df(name, df)


region_0
--------
shape: (100000, 5)
missing: 0
duplicates: 0
columns: ['id', 'f0', 'f1', 'f2', 'product']
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647

region_1
--------
shape: (100000, 5)
missing: 0
duplicates: 0
columns: ['id', 'f0', 'f1', 'f2', 'product']
      id         f0        f1        f2     product
0  kBEdx -15.001348 -8.276000 -0.005876    3.179103
1  62mP7  14.272088 -3.475083  0.999183   26.953261
2  vyE1P   6.263187 -5.948386  5.001160  134.766305

region_2
--------
shape: (100000, 5)
missing: 0
duplicates: 0
columns: ['id', 'f0', 'f1', 'f2', 'product']
      id        f0        f1        f2    product
0  fwXo0 -1.146987  0.963328 -0.828965  27.758673
1  WJtFt  0.262778  0.269839 -2.530187  56.069697
2  ovLUW  0.194587  0.289035 -5.586433  62.871910


**Interpretacion ejecutiva**
Los tres datasets tienen 100,000 registros y 5 columnas, sin faltantes ni duplicados. Esto reduce el riesgo de sesgo por calidad de datos y habilita comparaciones directas entre regiones.

## Preparacion de datos
Para el modelo se usan solo las variables numericas `f0`, `f1`, `f2`. El campo `id` es un identificador operativo y no aporta poder predictivo, por lo que se excluye.

In [7]:
FEATURES = ["f0", "f1", "f2"]
TARGET = "product"


def train_validate(df, test_size=0.25, random_state=RANDOM_STATE):
    X = df[FEATURES]
    y = df[TARGET]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_valid)
    mse = mean_squared_error(y_valid, preds)
    rmse = np.sqrt(mse)
    mean_pred = preds.mean()
    mean_target = y_valid.mean()

    return {
        "model": model,
        "X_valid": X_valid,
        "y_valid": y_valid,
        "preds": preds,
        "rmse": rmse,
        "mean_pred": mean_pred,
        "mean_target": mean_target,
    }

In [8]:
results = {name: train_validate(df) for name, df in data.items()}

summary = pd.DataFrame(
    {
        name: {
            "mean_pred": res["mean_pred"],
            "mean_target": res["mean_target"],
            "rmse": res["rmse"],
        }
        for name, res in results.items()
    }
).T

summary

,mean_pred,mean_target,rmse
region_0,92.592568,92.078597,37.579422
region_1,68.728547,68.723136,0.893099
region_2,94.965046,94.884233,40.029709


**Interpretacion ejecutiva**
- Calibracion: el promedio predicho es muy cercano al promedio real en las tres regiones, lo que indica que el modelo no esta sesgado.
- Incertidumbre: region_1 tiene el RMSE mas bajo (0.89), lo que implica alta estabilidad en la prediccion. En region_0 (37.58) y region_2 (40.03) la incertidumbre es mucho mayor.
- Implicacion: ningun promedio se acerca al umbral de 111.11, por lo que la rentabilidad dependera de seleccionar los pozos top en cada region.

## Lectura ejecutiva del modelo
La tabla resume dos señales clave para la toma de decision:
- **RMSE**: mide el error promedio de prediccion. Un RMSE mas bajo implica menor incertidumbre operativa.
- **Promedio predicho vs. promedio real**: confirma si el modelo esta calibrado y si la region, en promedio, se acerca al umbral minimo de rentabilidad.

En este punto no tomamos una decision final; solo verificamos si el modelo es suficientemente estable para apoyar la seleccion de pozos top.

In [9]:
for name, res in results.items():
    rmse = res["rmse"]
    mean_pred = res["mean_pred"]
    mean_target = res["mean_target"]
    gap = mean_pred - MIN_PRODUCT_PER_WELL

    print(f"\n{name}")
    print(f"  mean_pred: {mean_pred:.2f}")
    print(f"  mean_target: {mean_target:.2f}")
    print(f"  rmse: {rmse:.2f}")

    if gap >= 0:
        print("  Lectura: el promedio predicho supera el umbral minimo de rentabilidad.")
    else:
        print("  Lectura: el promedio predicho queda por debajo del umbral; se requiere seleccionar pozos top.")


region_0
  mean_pred: 92.59
  mean_target: 92.08
  rmse: 37.58
  Lectura: el promedio predicho queda por debajo del umbral; se requiere seleccionar pozos top.

region_1
  mean_pred: 68.73
  mean_target: 68.72
  rmse: 0.89
  Lectura: el promedio predicho queda por debajo del umbral; se requiere seleccionar pozos top.

region_2
  mean_pred: 94.97
  mean_target: 94.88
  rmse: 40.03
  Lectura: el promedio predicho queda por debajo del umbral; se requiere seleccionar pozos top.


## Preparacion para calculo de ganancias
El umbral minimo de rentabilidad por pozo es `MIN_PRODUCT_PER_WELL` (en miles de barriles). Comparamos ese umbral con el promedio de reservas reales por region para entender la presion operativa: si el promedio real esta por debajo, la rentabilidad depende de seleccionar los mejores pozos.

In [10]:
mean_product = {name: df[TARGET].mean() for name, df in data.items()}
mean_product = pd.Series(mean_product, name="mean_product")

mean_product

region_0    92.500
region_1    68.825
region_2    95.000
Name: mean_product, dtype: float64

**Interpretacion ejecutiva**
Los promedios reales por region (92.50, 68.83 y 95.00) estan por debajo del umbral de 111.11. Esto confirma que el exito financiero depende de una seleccion estricta de los 200 mejores pozos, no del promedio general.

## Funcion de ganancia y seleccion de 200 pozos
Calculamos la ganancia potencial seleccionando los 200 pozos con mayor prediccion. La ganancia usa reservas reales de esos pozos y descuenta el presupuesto total.

In [11]:
def calculate_profit(preds, actual, n_wells=N_WELLS, revenue_per_unit=REVENUE_PER_UNIT, budget=BUDGET):
    preds = np.asarray(preds)
    actual = np.asarray(actual)

    top_idx = np.argsort(preds)[::-1][:n_wells]
    total_product = actual[top_idx].sum()
    profit = total_product * revenue_per_unit - budget

    return profit, total_product, top_idx


top_200 = {}
profit_summary = {}

for name, res in results.items():
    profit, total_product, top_idx = calculate_profit(res["preds"], res["y_valid"])

    top_200[name] = {
        "preds": res["preds"][top_idx],
        "actual": res["y_valid"].to_numpy()[top_idx],
    }

    profit_summary[name] = {
        "total_product": total_product,
        "profit": profit,
    }

profit_table = pd.DataFrame(profit_summary).T
profit_table

,total_product,profit
region_0,29601.835651,3.320826e+07
region_1,27589.081548,2.415087e+07
region_2,28245.222141,2.710350e+07


In [12]:
profit_ranking = profit_table.sort_values("profit", ascending=False)
profit_ranking

,total_product,profit
region_0,29601.835651,3.320826e+07
region_2,28245.222141,2.710350e+07
region_1,27589.081548,2.415087e+07


**Interpretacion ejecutiva**
Si solo miramos beneficio potencial, la region_0 lidera (~$33.21M), seguida por region_2 (~$27.10M) y region_1 (~$24.15M). Esta lectura es preliminar porque aun no incorpora riesgo de perdida.

## Decision preliminar (solo beneficio, sin riesgo)
La tabla anterior indica la region con mayor beneficio potencial si solo consideramos las mejores predicciones. Esta lectura es util para priorizar, pero no es suficiente para decidir; falta cuantificar el riesgo de perdida.

## Analisis de riesgo con bootstrapping
Simulamos 1000 escenarios con muestras de 500 pozos por region, seleccionando los mejores 200 en cada simulacion. Esto estima el beneficio esperado, el intervalo de confianza y el riesgo de perdida.

In [13]:
def bootstrap_profit(preds, actual, n_boot=BOOTSTRAP_SAMPLES, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    preds = np.asarray(preds)
    actual = np.asarray(actual)

    profits = []
    for _ in range(n_boot):
        idx = rng.choice(len(preds), size=sample_size, replace=True)
        sample_preds = preds[idx]
        sample_actual = actual[idx]
        profit, _, _ = calculate_profit(sample_preds, sample_actual)
        profits.append(profit)

    return pd.Series(profits)


In [14]:
bootstrap_results = {}

for name, res in results.items():
    profits = bootstrap_profit(res["preds"], res["y_valid"])
    bootstrap_results[name] = {
        "mean_profit": profits.mean(),
        "ci_low": profits.quantile(0.025),
        "ci_high": profits.quantile(0.975),
        "loss_risk": (profits < 0).mean(),
    }

bootstrap_summary = pd.DataFrame(bootstrap_results).T
bootstrap_summary

,mean_profit,ci_low,ci_high,loss_risk
region_0,3.961650e+06,-1.112155e+06,9.097669e+06,0.069
region_1,4.560451e+06,3.382051e+05,8.522895e+06,0.015
region_2,4.044039e+06,-1.633504e+06,9.503596e+06,0.076


**Interpretacion ejecutiva**
- Solo region_1 cumple el criterio de riesgo (<2.5%), con 1.50% de probabilidad de perdida.
- region_0 y region_2 muestran riesgos de 6.90% y 7.60%, y sus limites inferiores del intervalo 95% son negativos, lo que indica escenarios plausibles de perdida.
- Aunque region_1 no era la mas alta en el ranking preliminar, su perfil riesgo-retorno es el unico aceptable bajo las reglas del negocio.

In [15]:
def format_money(value):
    return f"${value / 1e6:,.2f}M"

eligible = bootstrap_summary[bootstrap_summary["loss_risk"] < 0.025]

if eligible.empty:
    print("Ninguna region cumple con el riesgo maximo permitido (<2.5%).")
else:
    decision = eligible.sort_values("mean_profit", ascending=False).index[0]
    print(f"Region recomendada: {decision}")
    print("Resumen ejecutivo de riesgo y beneficio:")

    for name, row in bootstrap_summary.iterrows():
        print(
            f"  {name} | mean_profit: {format_money(row['mean_profit'])} | "
            f"CI95%: [{format_money(row['ci_low'])}, {format_money(row['ci_high'])}] | "
            f"loss_risk: {row['loss_risk']:.2%}"
        )

Region recomendada: region_1
Resumen ejecutivo de riesgo y beneficio:
  region_0 | mean_profit: $3.96M | CI95%: [$-1.11M, $9.10M] | loss_risk: 6.90%
  region_1 | mean_profit: $4.56M | CI95%: [$0.34M, $8.52M] | loss_risk: 1.50%
  region_2 | mean_profit: $4.04M | CI95%: [$-1.63M, $9.50M] | loss_risk: 7.60%


**Interpretacion ejecutiva**
La decision final es invertir en region_1. La region lidera en beneficio esperado ajustado por riesgo y es la unica que cumple el umbral de perdida permitido. La recomendacion cambia frente al ranking preliminar porque el riesgo operativo es el factor determinante.

## Conclusion ejecutiva
- La recomendacion final debe cumplir dos condiciones: riesgo de perdida < 2.5% y mayor beneficio promedio.
- La region recomendada aparece en la salida del bloque anterior junto con su intervalo de confianza, lo que permite validar la robustez de la decision.
- Si ninguna region cumple el riesgo, se recomienda no invertir o replantear el presupuesto y el numero de pozos.